## FastAPI Cheatsheet for ML/AI products deployment 

### What is an API? 
An **API (Application Programming Interface)** is a **middleman** that allows two systems to talk. A good mental model is that of a restaurant waiter, where 
- The user (frontend) looks at the menu (API docs)
- The waiter (API route) takes your order (HTTP request) to the kitchen (backend logic)
- The chef (model or code) processes it 
- The waiter then returns the dish (HTTP response)

APIs are required to **expose your model or function to others (especially non-technical users)** in a clean, standard way. 
- This allows apps, browsers, mobile devices, or even scripts to call your logic remotely. 
- Imagine a scenario where you've written the Python scripts that trains a model, saves it as model.pkl and is able to use this model to make predictions/inferences. 
    - However, without APIs, it is still stuck in my notebook or script
    - Nobody else can interact with my model unless they open my scripts/Jupyter notebook, understand Python and rerun everything manually 
    - This means that the code is not scalable, accessible, and hence not production ready
- FastAPI opens the door to the world by wrapping my model behind a REST API, so that 
    - Other people/apps/systems can send inputs to my model 
    - My API catches the request, loads the saved model, runs inference, and returns the results 
    - This happens in milliseconds (very fast)
- I.e. the model is the brain, but without the mouth and eyes/ears, no one can communicate to it 
    - FastAPI gives my model HTTP endpoints that function as its mouth and ears 
- NOTE: Flask and FastAPI serve the same high-level purpose, which is to turn Python code into web-accessible APIs. 
    - However, FastAPI is a modern evolution that is better suited for today's data/ML workflows 
- NOTE2: FastAPI automatically generates a fully functional frontend (docs interface) for your API. Dont need to write any HTML, JS, CSS to get 
    - Live interactive API playground (Swagger UI)
        - Can test endpoints live with buttons, dropdowns, and example payloads 
    - Clean dev reference site (ReDoc)
        - get clean, scrollable OpenAPI spec with request/response types

Key components of building FastAPI: 
- fastapi: web framework that allows exposing of Python functions (e.g. model inference) as web-accessible APIs 
    - Built on Starlette (fast, async-capable web layer)
    - Automatically generates Swagger and ReDoc from function signatures and type hints 
    - Enables definition of HTTP methods like POST, GET, PUT, DELETE using Python decorators directly on your functions 
        - I.e. I just need to write a regular Python function (e.g. predict) and wrap it in the Python decorator (e.g. @app.post("/predict")) to define HTTP methods 
- uvicorn: lightning-fast server that runs the FastAPI app
    - Built on the ASGI: Asynchronous Server Gateway Interface which is the modern replacement for WSGI 
    - Runs with uvicorn main:app --reload 
- pydantic: validates and parses the input data
- joblib: for loading the saved model 

Mental Model 
- FastAPI designs the car, Uvicorn makes it drive under the hood (engine)
- Pydantic is the form-checking front desk that filters out bad inputs from reaching and crashing the model 
- Joblib is the USB stick that stores the trained model, which FastAPI uses to plug in and serve the requests 

Supported HTTP Methods in FastAPI
- GET: retrieves info (read only)
    - @app.get("/...")
- POST: submits data for processing 
    - @app.post("/...")
- PUT: updates a full resource 
    - @app.put("/...")
- PATCH: partially update a resource 
    - @app.patch("/...")
- DELETE: delete a resource 
    - @app.delete("/...")

Paired with functions that support type hints (to say what the inputs and outputs should be)
- E.g. def add(a,b): <here, we don't know what a and b should be. int? str? arrows?>
        - return a + b 
- With type hints, we remove this ambiguity e.g. def add(a: int, b: int) -> int: 
        0 return a + b 
- In FastAPI, when working with JSON request bodies, define type hints using Pydantic's BaseModel (see below)
- NOTE: function names must be unique within a Python file, and route paths must be unique within the FastAPI app

Sample HTTP POST Request
- POST /predict HTTP/1.1               ← HTTP method (POST), endpoint (calling the /predict route), and version
- Content-Type: application/json       ← Declares to the server that the body format is JSON

- {
  - "feature1": 1.0,                   ← The actual request body (JSON)
  - "feature2": 2.0
- }


In [ ]:
#Step 0: Save model 
import joblib 
joblib.dump(model, "model.pkl")

In [ ]:
#Step 1: Create API and load model 
from fastapi import FastAPI
from pydantic import BaseModel 
from typing import Union, List
import joblib 

app = FastAPI()
model = joblib.load("model.pkl")

#Step 2: Use Pydantic to define complex input structures (type hints)
class InputData(BaseModel): 
    feature1: float
    feature2: float

#Step 3: Define endpoints (e.g. predict) for batch prediction
@app.post("/predict")
def predict(data: List[InputData]): #use the said type hint in the POST endpoint, but wrap it into a list. Triggers auto-validation via the Pydantic model 
    X = [[item.feature1, item.feature2] for item in data] #use list comprehension to unpack. double list since most modern ML models (e.g. sklearn) expect 2D arrays corresponding to n_obs * n_features. This is since everything is batched, so even when predicting single input, need to wrap it as a batch of 1
    preds = model.predict(X) #model.predict(X) returns a NumPy array even if only send in one row, so pred[0] returns the actual scalar value
    return {"prediction": [float(p) for p in preds]} #best practice is to return a dict and FastAPI will convert into JSON automatically (JSON is the standard data format for REST APIs)

#Step 3b: Define endpoint that can auto-handle single or batch input 
@app.post("predict_auto/", summary="Auto-handling of single or batch predictions", description="Takes inputs and returns predictions using model")
def predict_auto(data: Union[InputData, List[InputData]] = Body(...)): #data: parameter name, what FastAPI will look for in the request body. Union[] is the type hint: telling FastAPI I'm expecting a single object OR a list of such objects. Union is from the typing module and means that this parameter can be one of multiple types. Body is a class from fastapi that tells it that this variable should be parsed from the request body and not URL path or query string
    #Check if its single input or list 
    if isinstance(data, InputData): 
        X = [[data.feature1, data.feature2]] #single input as batch 
        preds = model.predict(X)
        return {"prediction": float(preds[0])}
    else: 
        X = [[item.feature1, item.feature2] for item in data]
        preds = model.predict(X)
        return {"predictions": [float(p) for p in preds]}
    
#Step 3c: Define other end points 
## Get model metadata or schema 
@app.get("/model-info")
def model_info():
    return {
        "model": "RandomForestClassifier",
        "version": "1.0.3",
        "input_features": ["feature1", "feature2"]
    }

##### Step 4: Run the App 
[Bash/Terminal] uvicorn main:app --reload
- main: name of the Python file (without .py)
- app: the FastAPI app instance 
- --reload: hot reloads on save; i.e. the server restarts automatically whenever make code changes (great for dev)

For production deployment, run uvicorn main:app --host 0.0.0.0 --port 8000 --workers 4
- --host 0.0.0.0 → Listens on all network interfaces (required for external access, e.g. in Docker or cloud)

- --port 8000 → Port to expose the app (you can change this to any open port)

- --workers 4 → Run with 4 parallel worker processes for better performance (adjust based on your CPU cores)



##### Step 5: Test it 
[Bash/Terminal] curl -X POST "http://127.0.0.1:8000/predict" \
     -H "Content-Type: application/json" \
     -d '{"feature1": 3.2, "feature2": 1.8}'


OR 

[Python] 
```python
import requests  # HTTP library for sending web requests from Python

# Send a POST request to the FastAPI endpoint
response = requests.post(
    "http://127.0.0.1:8000/predict",  # URL of your FastAPI endpoint
    json={"feature1": 3.2, "feature2": 1.8}  # Data to send in JSON format
)

# Parse and print the JSON response as a Python dictionary
print(response.json())